# Lab 04 (Colab): Perplexity, cross-entropy, and bits-per-byte

**Chapter 3, "Evaluation Methodology"**, section *Understanding Language Modeling Metrics* (entropy, cross entropy, bits-per-character and bits-per-byte, perplexity, perplexity interpretation and use cases)
**Environment:** Google Colab, free **T4 GPU** | **Time:** about 1.5-2 hours (the cluster notebook is the other part of this lab)

Before Larkspur trusts any model with its documents, it helps to know what the model's most basic score means. A language model assigns a probability to every next token. **Cross-entropy** is how surprised the model is on average, **perplexity** turns that into "how many equally likely choices it was torn between", and **bits-per-byte** makes the number comparable across tokenizers. You'll compute all three by hand from the model's logits, on a Larkspur runbook, on scrambled and random text, and on a famous public-domain book, and see how the numbers change with the model and tokenizer.

**Before you start**
- Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all is safe: every TODO has a placeholder.
- This notebook downloads four small models (about 7 GB in total; a few minutes on Colab). It does not use the course cluster (Colab can't reach it).

## Setup

In [ ]:
!pip -q install "itec632 @ git+https://github.com/tyler-schunk/itec632-course#subdirectory=image"

In [ ]:
import gc, math, random, string, time

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from itec632.grading import LabChecker

ck = LabChecker("lab04", part="colab")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: Runtime > Change runtime type > T4 GPU)")

In [ ]:
def load_model(name):
    """Load a tokenizer and causal LM in float32 (exact math; these models are small enough for a T4)."""
    tok = AutoTokenizer.from_pretrained(name)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    except TypeError:                       # older transformers used torch_dtype=
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32)
    return tok, model.to(DEVICE).eval()


def free_gpu():
    """Release GPU memory after you `del` a model."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def encode(tok, text):
    """Token ids for text, with a document-start token in front so that EVERY token of the text gets predicted."""
    start = tok.convert_tokens_to_ids("<|endoftext|>")
    if start is None or start == tok.unk_token_id:
        start = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    ids = [start] + tok(text, add_special_tokens=False)["input_ids"]
    return torch.tensor([ids], device=DEVICE)


MAIN = "Qwen/Qwen3-0.6B"
tok, model = load_model(MAIN)
print(MAIN, "loaded;", sum(p.numel() for p in model.parameters()) / 1e6, "M parameters; vocab", len(tok))

# Part A - Guided practice

## A1. Tokens and bytes

Our main text is the opening of Larkspur's current runbook **RB-PRT-001 "Printer shows offline"** (v2). A tokenizer splits text into tokens; the model's metrics are *per token*, but people care about *text*, so we also count UTF-8 bytes.

**TODO:** compute how many bytes of text each token covers on average.

In [ ]:
PASSAGE = 'Use when a user reports a printer shows **Offline** in Windows, jobs sit in the queue and never print, or "Error - Printing" won\'t clear. Applies to network HP and Brother printers/MFPs at client sites, printed to directly from Windows 10/11 PCs via a Standard TCP/IP port. If the printer is shared from a client print server, see the print server note in step 9.\n\nWork the steps **in order**. Most "offline" printers are either a network/IP problem or a stuck spool, and clearing the spool before confirming the printer is reachable just wastes time.\n\n\n\n## Steps\n\n### 1. Check power and network first\n\n1. Ask the user (or a coworker near the printer) to look at the printer:\n   - Is it powered on and awake? Press a button to wake it from sleep.\n   - Any error on the front panel (paper jam, toner, "door open", "IP conflict")? Clear physical errors first; jams and empty trays often show as "offline" or "error" in Windows.\n   - Is the network cable plugged in with link lights? On Wi-Fi printers, is the Wi-Fi indicator solid?\n2. **Find the printer\'s current IP** from its front panel (HP: Settings/Network or the network icon; Brother: Menu > Network > Wired LAN or WLAN > TCP/IP). Compare with the IP in the client\'s Atlas Docs printer list and the IP on the Windows port:\n   - On the user\'s PC: Settings > Bluetooth & devices > Printers & scanners > the printer > **Printer properties** > **Ports** tab, note the checked port and its address (Configure Port).'

ids = encode(tok, PASSAGE)
n_tokens = ids.shape[1] - 1                       # minus the start token
n_bytes = len(PASSAGE.encode("utf-8"))
print(n_tokens, "tokens,", n_bytes, "bytes")
print([tok.decode([t]) for t in ids[0, 1:25].tolist()])

In [ ]:
bytes_per_token = None     # TODO: bytes divided by tokens
print("bytes per token:", bytes_per_token)

In [ ]:
ck.check("A1_bytes_per_token", bytes_per_token is not None and abs(bytes_per_token - n_bytes / n_tokens) < 1e-9
         and 2 < bytes_per_token < 8, points=2, hint="bytes_per_token = n_bytes / n_tokens (English text is usually 3-5 bytes per token).")

## A2. Cross-entropy from the logits

For every position the model outputs **logits**, one score per vocabulary token. `log_softmax` turns them into log-probabilities. The model's loss on a token is the negative log-probability it gave to the token that actually came next. Average that over the text and you have the **cross-entropy** in nats (natural log). Hugging Face computes the same number when you pass `labels=`; you'll reproduce it by hand.

Note the shift: the logits at position *i* predict token *i+1*.

**TODO:** compute `token_nll`, the negative log-probability of each actual next token.

In [ ]:
with torch.no_grad():
    out = model(ids, labels=ids)
logits = out.logits[0, :-1].float()      # predictions for positions 1..n
targets = ids[0, 1:]                     # the tokens that actually came next

# TODO: logprobs = torch.log_softmax(logits, dim=-1)
# TODO: token_nll = -(log-probability of each target), shape [n_tokens]  (hint: logprobs.gather(1, targets[:, None]).squeeze(1))
token_nll = None

print("HF loss (nats/token):", round(out.loss.item(), 4), "  by hand:", None if token_nll is None else round(token_nll.mean().item(), 4))

In [ ]:
ok = token_nll is not None and token_nll.shape[0] == n_tokens and abs(token_nll.mean().item() - out.loss.item()) < 1e-3
ck.check("A2_cross_entropy_by_hand", ok, points=3,
         hint="Your per-token NLL should have one value per text token and average to the HF loss. Did you negate the gathered log-probs?")
if token_nll is not None:
    worst = torch.topk(token_nll, 8)
    print("Most surprising tokens (the model did not expect these):")
    for v, i in zip(worst.values.tolist(), worst.indices.tolist()):
        print(f"  {tok.decode([targets[i].item()])!r:20} {v:.2f} nats  (p = {math.exp(-v):.4f})")

## A3. Perplexity and bits-per-byte

The chapter defines these from cross-entropy *H*:
- **bits per token** = *H* in nats / ln 2 (bits use log base 2),
- **perplexity** = e^*H* (equivalently 2^bits-per-token): the model is, on average, as uncertain as if it were picking uniformly among this many tokens,
- **bits per byte (BPB)** = total bits over the whole text / number of bytes. Unlike perplexity, BPB does not depend on how the tokenizer cuts the text, so you can compare models with different tokenizers.

**TODO:** finish `lm_metrics()`.

In [ ]:
def lm_metrics(tok, model, text):
    """Cross-entropy, bits per token, perplexity, and bits per byte of `text` under `model`."""
    ids = encode(tok, text)
    with torch.no_grad():
        logits = model(ids).logits[0, :-1].float()
    nll = -torch.log_softmax(logits, dim=-1).gather(1, ids[0, 1:, None]).squeeze(1)   # nats per token
    n_tok, n_bytes = nll.numel(), len(text.encode("utf-8"))
    ce_nats = nll.mean().item()
    bits_per_token = None      # TODO: convert nats to bits (divide by math.log(2))
    perplexity = None          # TODO: math.exp(ce_nats)
    bpb = None                 # TODO: TOTAL bits for the text (nll.sum() in bits) divided by n_bytes
    return {"tokens": n_tok, "bytes": n_bytes, "ce_nats": ce_nats, "bits_per_token": bits_per_token,
            "perplexity": perplexity, "bpb": bpb}


m = lm_metrics(tok, model, PASSAGE)
print(m)

In [ ]:
ok = (m["perplexity"] is not None and m["bpb"] is not None and m["bits_per_token"] is not None
      and abs(m["perplexity"] - math.exp(m["ce_nats"])) < 1e-6
      and abs(m["bpb"] - m["bits_per_token"] * m["tokens"] / m["bytes"]) < 1e-6
      and 1 < m["perplexity"] < 1000)
ck.check("A3_metrics", ok, points=3, hint="perplexity = exp(ce_nats); bits_per_token = ce_nats / ln 2; bpb = total bits / bytes.")
ck.record("A3_runbook_ppl_qwen3_0.6b", m["perplexity"])
ck.record("A3_runbook_bpb_qwen3_0.6b", m["bpb"])

# Part B - Applied lab

## B1. Structured text vs. scrambled vs. random vs. a famous book

The chapter explains that perplexity is lower for more **structured, predictable** text, and that a model can have suspiciously low perplexity on text it saw (or memorized) in training. Let's test that with five texts:

1. the Larkspur runbook passage,
2. the same passage with its words shuffled (same words, no structure),
3. random characters of the same length,
4. the opening of *Pride and Prejudice* (1813, public domain, and almost certainly in every model's training data),
5. **your own text** (TODO): write or paste a paragraph of your own (at least 300 characters). Something you wrote yourself and never posted online is most interesting.

In [ ]:
rng = random.Random(632)
words = PASSAGE.split()
SHUFFLED = " ".join(rng.sample(words, len(words)))
RANDOM_CHARS = "".join(rng.choice(string.ascii_letters + string.digits + string.punctuation + " ") for _ in range(len(PASSAGE)))
PRIDE = 'It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.\n\nHowever little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.\n\n"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"\n\nMr. Bennet replied that he had not.\n\n"But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."\n\nMr. Bennet made no answer.\n\n"Do you not want to know who has taken it?" cried his wife impatiently.\n\n"You want to tell me, and I have no objection to hearing it."\n\nThis was invitation enough.'

In [ ]:
MY_TEXT = ""      # TODO: your own paragraph (at least 300 characters)

In [ ]:
TEXTS = {"runbook": PASSAGE, "shuffled": SHUFFLED, "random_chars": RANDOM_CHARS, "pride_prejudice": PRIDE}
if len(MY_TEXT) >= 300:
    TEXTS["my_text"] = MY_TEXT
rows = []
for name, text in TEXTS.items():
    r = lm_metrics(tok, model, text)
    rows.append({"text": name, **r})
b1 = pd.DataFrame(rows).set_index("text")
display(b1.round(3))
print("For comparison: uniformly random characters from this 95-symbol alphabet carry log2(95) =", round(math.log2(95), 2), "bits per byte.")

In [ ]:
ppl = b1["perplexity"] if b1["perplexity"].notna().all() else None
ck.check("B1_table", ppl is not None and len(b1) >= 4, points=2, hint="lm_metrics() must return numbers for every text (finish A3).")
ck.check("B1_structure_ordering", ppl is not None and ppl["random_chars"] > ppl["shuffled"] > ppl["runbook"], points=2,
         hint="Expected random_chars > shuffled > runbook. If not, check that SHUFFLED/RANDOM_CHARS were built and lm_metrics is right.")
ck.check("B1_memorized_text_is_easy", ppl is not None and ppl["pride_prejudice"] < ppl["shuffled"], points=1,
         hint="The famous book passage should be far more predictable than shuffled words.")
ck.check("B1_my_text", "my_text" in TEXTS, points=1, hint="Set MY_TEXT to a paragraph of at least 300 characters.")
if ppl is not None:
    for name in b1.index:
        ck.record(f"B1_ppl_{name}", float(b1.loc[name, "perplexity"]))
        ck.record(f"B1_bpb_{name}", float(b1.loc[name, "bpb"]))

## B2. Perplexity depends on the model and the tokenizer

Perplexity is not a property of the text alone. Compare four small models on the runbook and on *Pride and Prejudice*:

| Model | What's different |
|---|---|
| `Qwen/Qwen3-0.6B-Base` | pre-trained only |
| `Qwen/Qwen3-0.6B` | same size and tokenizer, **post-trained** (instruction tuning, preference tuning) |
| `Qwen/Qwen3-1.7B` | same family and tokenizer, about 3x larger |
| `HuggingFaceTB/SmolLM2-360M` | different family and a **different tokenizer** (about 49k tokens vs Qwen's 151k) |

**TODO:** inside the loop, compute `lm_metrics` for each text and append a row `{"model": name, "text": text_name, **metrics}`. (About 5 minutes, mostly downloads.)

In [ ]:
del model          # free the GPU before loading the next models
free_gpu()
MODELS_B2 = ["Qwen/Qwen3-0.6B-Base", "Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "HuggingFaceTB/SmolLM2-360M"]
B2_TEXTS = {"runbook": PASSAGE, "pride_prejudice": PRIDE}
b2_rows = []

In [ ]:
for name in MODELS_B2:
    t0 = time.time()
    tok_i, model_i = load_model(name)
    for text_name, text in B2_TEXTS.items():
        pass   # TODO: b2_rows.append({"model": name, "text": text_name, **lm_metrics(tok_i, model_i, text)})
    print(f"{name}: done in {time.time() - t0:.0f}s")
    del model_i
    free_gpu()

In [ ]:
b2 = pd.DataFrame(b2_rows)
if len(b2):
    display(b2.pivot(index="model", columns="text", values=["tokens", "perplexity", "bpb"]).round(3))
ok_table = len(b2) == 8 and b2["perplexity"].notna().all()
ck.check("B2_table", ok_table, points=3, hint="You need 4 models x 2 texts = 8 rows with perplexity and bpb.")
tok_effect = False
if ok_table:
    run = b2[b2["text"] == "runbook"].set_index("model")
    qwen_tokens = run.loc[[m for m in MODELS_B2 if m.startswith("Qwen")], "tokens"]
    tok_effect = qwen_tokens.nunique() == 1 and run.loc["HuggingFaceTB/SmolLM2-360M", "tokens"] != qwen_tokens.iloc[0]
ck.check("B2_tokenizer_effect", tok_effect, points=1,
         hint="The three Qwen models share a tokenizer (same token count); SmolLM2 should count differently.")
if ok_table:
    for mname in MODELS_B2:
        short = mname.split("/")[-1]
        ck.record(f"B2_runbook_ppl_{short}", float(run.loc[mname, "perplexity"]))
        ck.record(f"B2_runbook_bpb_{short}", float(run.loc[mname, "bpb"]))

## B3. More context, lower perplexity

The chapter notes that the more context a model has, the less uncertain it is about what comes next. Measure the perplexity of the **second half** of the runbook passage twice: on its own, and with the first half in front of it as context (only the second half is scored).

To score only part of the input, Hugging Face lets you set a label to `-100`; those positions are ignored in the loss.

**TODO:** build `labels`: `-100` for the start token and every context token, then the real ids of the text tokens.

In [ ]:
tok, model = load_model(MAIN)
cut = PASSAGE.index("\n", len(PASSAGE) // 2)          # split at a line break near the middle
FIRST, SECOND = PASSAGE[:cut], PASSAGE[cut:]
start_id = encode(tok, "")[0, 0].item()
ctx_ids = tok(FIRST, add_special_tokens=False)["input_ids"]
txt_ids = tok(SECOND, add_special_tokens=False)["input_ids"]

In [ ]:
def conditional_ppl(context_ids, text_ids):
    input_ids = [start_id] + context_ids + text_ids
    labels = None   # TODO: [-100] * (1 + len(context_ids)) + text_ids
    if labels is None:
        return None
    with torch.no_grad():
        loss = model(torch.tensor([input_ids], device=DEVICE), labels=torch.tensor([labels], device=DEVICE)).loss
    return math.exp(loss.item())

In [ ]:
ppl_alone = conditional_ppl([], txt_ids)
ppl_with_context = conditional_ppl(ctx_ids, txt_ids)
print("second half alone:", ppl_alone, "   with the first half as context:", ppl_with_context)
ck.check("B3_context_lowers_ppl", ppl_alone is not None and ppl_with_context is not None and ppl_with_context < ppl_alone,
         points=2, hint="Build labels so only the second half is scored; with context its perplexity should drop.")
ck.record("B3_ppl_alone", ppl_alone)
ck.record("B3_ppl_with_context", ppl_with_context)

# Written analysis

Use **your numbers** from this notebook. Cite the chapter by section name where asked.

**Q1. Structure and predictability.** Rank your texts by perplexity and by bits-per-byte (B1). Why is the shuffled runbook so much harder than the original, even though it has exactly the same words? How does the random-character BPB compare with the 6.57 bits of a uniform choice among 95 symbols, and why can't a model do much better on it? Use the definitions in *Understanding Language Modeling Metrics* (entropy, cross entropy, perplexity).

*Your answer:*

**Q2. The famous book.** How did *Pride and Prejudice* compare with your own text and the runbook? The chapter lists uses of perplexity, including detecting whether a text was likely in a model's training data (*Perplexity Interpretation and Use Cases*). What does your result suggest, and why does this matter when you read a model's score on a public benchmark (connect to data contamination)?

*Your answer:*

**Q3. Model and tokenizer.** From B2: why can't you compare SmolLM2's perplexity with Qwen's directly, while bits-per-byte is comparable? Which model has the lowest BPB on the runbook, and is that what you expected from model size? Did post-training (Base vs. the post-trained Qwen3-0.6B) raise or lower perplexity, and how does the chapter explain that direction?

*Your answer:*

**Q4. Is perplexity useful at Larkspur?** Give one concrete way perplexity could help the help-desk assistant project (for example, flagging garbled ticket text, or filtering data before fine-tuning in Lab 10) and one thing it can **not** tell you about an answer (hint: an outdated runbook and the current one can have similar perplexity).

*Your answer:*

# AI-use reflection

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Your metric code was checked against Hugging Face's own loss in A2. If you had no such reference, how would you have convinced yourself the perplexity formula was right?

*Your answer:*

## Submit

Run this last, save (File > Download > .ipynb), upload the .ipynb.

In [ ]:
ck.report()